## DeepFashion In-Shop folder structure

This project uses the official **DeepFashion In-Shop Clothes Retrieval** dataset.

### Main folders
- **`Img/`**  
  Contains the product images. After extraction, images are typically under `img/...` with category/subfolder structure such as `MEN/` and `WOMEN/`.

- **`Anno/`**  
  Contains annotations used for training and preprocessing:
  - `list_bbox_inshop.txt`: bounding boxes for each image
  - `list_item_inshop.txt`: item IDs
  - `list_description_inshop.json`: item descriptions
  - other files for landmarks, attributes, segmentation, etc.

- **`Eval/`**  
  Contains `list_eval_partition.txt`, which defines the official split:
  - `train`: used for model training / fine-tuning
  - `query`: used as search queries during evaluation
  - `gallery`: database searched during evaluation

- **`README.txt`**  
  Official dataset documentation.

### How these are used in our pipeline
- **YOLO**  
  Used to detect/crop the main clothing item from each image. This can be compared against the provided bounding boxes in `Anno/list_bbox_inshop.txt`.

- **CLIP**  
  Encodes images into embeddings for retrieval.  
  - Training images from `Eval/list_eval_partition.txt` with status `train` are used for fine-tuning.
  - Query and gallery images are used for retrieval evaluation.

- **BLIP / BLIP-2**  
  Generates captions or semantic descriptions for product images.  
  These captions can be fused with CLIP image embeddings to improve retrieval.

### Retrieval setup
For evaluation, each **query** image is matched against the **gallery** set.  
A retrieval is considered correct when the query and retrieved image share the same **item ID**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

root = "/content/drive/MyDrive/VR_project/In-shop Clothes Retrieval Benchmark"

for name in os.listdir(root):
    p = os.path.join(root, name)
    print(name, "DIR" if os.path.isdir(p) else "FILE")

Eval DIR
Img DIR
Anno DIR
README.txt FILE


In [ ]:
img_dir = os.path.join(root, "Img")
print(os.listdir(img_dir)[:20])

['img_highres.zip', 'img.zip']


In [ ]:
import os

root = "/content/drive/MyDrive/VR_project/In-shop Clothes Retrieval Benchmark"
zip_path = os.path.join(root, "Img", "img.zip")

print("Exists:", os.path.exists(zip_path))
print("Size (GB):", os.path.getsize(zip_path) / (1024**3))

Exists: True
Size (GB): 0.7735066078603268


In [ ]:
import os

shared_root = "/content/drive/MyDrive/VR_project/In-shop Clothes Retrieval Benchmark"
my_copy_root = "/content/drive/MyDrive/VR_project/deepfashion_inshop_working"

os.makedirs(my_copy_root, exist_ok=True)
os.makedirs(os.path.join(my_copy_root, "Img"), exist_ok=True)

zip_path = os.path.join(shared_root, "Img", "img.zip")
out_dir = os.path.join(my_copy_root, "Img")

!unzip -q "$zip_path" -d "$out_dir"
print("Done")

Done


In [ ]:
import os

base = "/content/drive/MyDrive/VR_project/deepfashion_inshop_working"
print("top:", os.listdir(base))
print("Img/img contents:", os.listdir(os.path.join(base, "Img/img"))[:10])

top: ['Img']
Img/img contents: ['MEN', 'WOMEN']


In [ ]:
import os, shutil

shared_root = "/content/drive/MyDrive/VR_project/In-shop Clothes Retrieval Benchmark"
my_copy_root = "/content/drive/MyDrive/VR_project/deepfashion_inshop_working"

for name in ["Anno", "Eval", "README.txt"]:
    src = os.path.join(shared_root, name)
    dst = os.path.join(my_copy_root, name)

    if os.path.isdir(src):
        if not os.path.exists(dst):
            shutil.copytree(src, dst)
            print("Copied dir:", name)
        else:
            print("Already exists:", name)
    else:
        shutil.copy2(src, dst)
        print("Copied file:", name)

print("\nTop level now:", os.listdir(my_copy_root))

Copied dir: Anno
Copied dir: Eval
Copied file: README.txt

Top level now: ['Img', 'Anno', 'Eval', 'README.txt']


In [ ]:
import os

root = "/content/drive/MyDrive/VR_project/deepfashion_inshop_working"

checks = [
    os.path.join(root, "Eval", "list_eval_partition.txt"),
    os.path.join(root, "Anno", "list_bbox_inshop.txt"),
    os.path.join(root, "Anno", "list_item_inshop.txt"),
    os.path.join(root, "Img", "img"),
]

for p in checks:
    print(p, "->", os.path.exists(p))

/content/drive/MyDrive/VR_project/deepfashion_inshop_working/Eval/list_eval_partition.txt -> True
/content/drive/MyDrive/VR_project/deepfashion_inshop_working/Anno/list_bbox_inshop.txt -> True
/content/drive/MyDrive/VR_project/deepfashion_inshop_working/Anno/list_item_inshop.txt -> True
/content/drive/MyDrive/VR_project/deepfashion_inshop_working/Img/img -> True


In [ ]:
import os

img_root = "/content/drive/MyDrive/VR_project/deepfashion_inshop_working/Img/img"

count = 0
for _, _, files in os.walk(img_root):
    count += sum(f.lower().endswith(".jpg") for f in files)

print("jpg_count =", count)

jpg_count = 52712


In [ ]:
import pandas as pd

eval_path = "/content/drive/MyDrive/VR_project/deepfashion_inshop_working/Eval/list_eval_partition.txt"

df = pd.read_csv(
    eval_path,
    sep=r"\s+",
    skiprows=2,
    header=None,
    names=["image_name", "item_id", "split"]
)

print(df.head())
print("\nSplit counts:")
print(df["split"].value_counts())

print("\nUnique item_ids per split:")
print(df.groupby("split")["item_id"].nunique())

                                          image_name      item_id  split
0       img/WOMEN/Dresses/id_00000002/02_1_front.jpg  id_00000002  train
1        img/WOMEN/Dresses/id_00000002/02_2_side.jpg  id_00000002  train
2        img/WOMEN/Dresses/id_00000002/02_4_full.jpg  id_00000002  train
3  img/WOMEN/Dresses/id_00000002/02_7_additional.jpg  id_00000002  train
4        img/WOMEN/Skirts/id_00000003/02_1_front.jpg  id_00000003  train

Split counts:
split
train      25882
query      14218
gallery    12612
Name: count, dtype: int64

Unique item_ids per split:
split
gallery    3985
query      3985
train      3997
Name: item_id, dtype: int64


Perfect. That tells us the split is healthy and the setup is real.

## What this means

* **Train**: 25,882 images, **3,997 items**
* **Query**: 14,218 images, **3,985 items**
* **Gallery**: 12,612 images, **3,985 items**

So the benchmark works like this:

* train on **train**
* evaluate by taking each **query** image and retrieving from **gallery**
* a result is correct if retrieved image has the same `item_id` as the query image

And yes, there is **no official validation split** here. If later you want validation for fine-tuning, we can carve a small one out of `train`. For final reported results, use **query vs gallery**.

## What to do next

Now we should do the **first true setup notebook step**:

### Build one master dataframe

Join:

* `Eval/list_eval_partition.txt`
* image absolute path
* bbox from `Anno/list_bbox_inshop.txt`

That gives you one table you can use for:

* EDA
* cropping
* dataloaders
* retrieval evaluation



In [ ]:
import pandas as pd
import os

root = "/content/drive/MyDrive/VR_project/deepfashion_inshop_working"

# eval file
eval_df = pd.read_csv(
    os.path.join(root, "Eval", "list_eval_partition.txt"),
    sep=r"\s+",
    skiprows=2,
    header=None,
    names=["image_name", "item_id", "split"]
)

# bbox file
bbox_df = pd.read_csv(
    os.path.join(root, "Anno", "list_bbox_inshop.txt"),
    sep=r"\s+",
    skiprows=2,
    header=None,
    names=["image_name", "clothes_type", "pose_type", "x1", "y1", "x2", "y2"]
)

# merge
df = eval_df.merge(bbox_df, on="image_name", how="inner")

# absolute path
df["image_path"] = df["image_name"].apply(lambda x: os.path.join(root, "Img", x))

print("merged shape:", df.shape)
print("missing paths:", (~df["image_path"].map(os.path.exists)).sum())
print(df.head())

merged shape: (52712, 10)
missing paths: 0
                                          image_name      item_id  split  \
0       img/WOMEN/Dresses/id_00000002/02_1_front.jpg  id_00000002  train   
1        img/WOMEN/Dresses/id_00000002/02_2_side.jpg  id_00000002  train   
2        img/WOMEN/Dresses/id_00000002/02_4_full.jpg  id_00000002  train   
3  img/WOMEN/Dresses/id_00000002/02_7_additional.jpg  id_00000002  train   
4        img/WOMEN/Skirts/id_00000003/02_1_front.jpg  id_00000003  train   

   clothes_type  pose_type   x1   y1   x2   y2  \
0             3          1   65   45  233  252   
1             3          2  112   41  168  247   
2             3          4   89   34  169  167   
3             3          5   73   40  194  251   
4             2          1   51  122  160  210   

                                          image_path  
0  /content/drive/MyDrive/VR_project/deepfashion_...  
1  /content/drive/MyDrive/VR_project/deepfashion_...  
2  /content/drive/MyDrive/VR_proje

In [ ]:
print(i for i in range(10))

<generator object <genexpr> at 0x7b3866ac2140>


In [ ]:
import os

root = "/content/drive/MyDrive/VR_project/deepfashion_inshop_working"

for name in ["Img", "Anno", "Eval", "README.txt"]:
    p = os.path.join(root, name)
    print(name, os.path.exists(p))
    if os.path.isdir(p):
        print("  sample:", os.listdir(p)[:5])

Img True
  sample: ['img']
Anno True
  sample: ['attributes', 'densepose', 'segmentation', 'list_bbox_inshop.txt', 'list_item_inshop.txt']
Eval True
  sample: ['list_eval_partition.txt']
README.txt True
